In [41]:
import torch
import pandas as pd
import numpy
import re
from collections import defaultdict

def get_indiv_concepts(formula) -> list:
    concepts = []
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        if ':tag:' in c[:end_idx] or 'oth:over' in c[:end_idx]:
            continue
        concepts.append(c[:end_idx])
    return concepts


def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
    
    return unit_concepts

perunit_concepts= []
for cluster in range(1,4):
    filepath= f'/workspace/CCE_NLI/LLAMA/exp/lottery_ticket/Run0.25_5/Expls/0.0%Pruned/Cluster{cluster}IOUS1024N.csv'
    perunit_concepts.append(load_csv_data(filepath))
filepath= f'/workspace/CCE_NLI/LLAMA/exp/singlecuster/Expls/ClusterNoneIOUS1024N.csv'
perunit_concepts.append(load_csv_data(filepath))
    

In [42]:
no_clustering_concepts=perunit_concepts[-1]

In [43]:
avg=0
aligned_neuron=defaultdict(set)
for neuron in no_clustering_concepts:
    count = 0
    for concept in no_clustering_concepts[neuron]:
        if concept in perunit_concepts[0][neuron] or concept in perunit_concepts[1][neuron] or concept in perunit_concepts[2][neuron]:
            count += 1
            aligned_neuron[neuron].add(concept)
            
    if len(no_clustering_concepts[neuron]) > 0:
        avg += (count/len(no_clustering_concepts[neuron]))      

In [44]:
len(aligned_neuron)/len(no_clustering_concepts) 
#how many neruons have a shared concpet with the clustered  (134/690) 
#so basiaclly we see that there are a lot of variation between the clustered and non clsuterd 
#so having the clusters lets us see more fine grained knowledge since even across clusters we can see different expls

0.19718309859154928

In [45]:
len(no_clustering_concepts) 

781

In [38]:
lenn=0
for neuron in no_clustering_concepts:

    lenn += len(no_clustering_concepts[neuron])
#this says how many no tags,oths are there in the no cluster expls

In [39]:
lenn/len(no_clustering_concepts) #this says that

0.46956521739130436